<h2><font color="#004D7F" size=5>Módulo 4: Stacking</font></h2>


<h1><font color="#004D7F" size=6> 4. Stacking</font></h1>
<br><br>
<div style="text-align: right">
<font color="#004D7F" size=3>Manuel Castillo-Cara</font><br>
<font color="#004D7F" size=3>Aprendizaje Automático II</font><br>
<font color="#004D7F" size=3>Universidad Nacional de Educación a Distancia</font>

</div>

---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>


* [1. Stacked generalization](#section1)
* [2. Especificaciones sobre Stacking](#section2)
* [3. Stacking para Clasificación](#section3)
    * [3.1. Resultado de línea base](#section31)
    * [3.2. Integración Stacking](#section32)
* [3. Stacking para Clasificación](#section3)
    * [3.1. Resultado de línea base](#section31)
    * [3.2. Integración Stacking](#section32)
* [Preguntas frecuentes](#sectionPr)
* [Ejercicios](#sectionEj)

---

<a id="section0"></a>
# <font color="#004D7F">0. Contexto</font>

Stacking utiliza un algoritmo de metaaprendizaje para aprender cómo combinar mejor las predicciones de dos o más algoritmos. En este tutorial, trabajaremos:
- Stacking aprende cómo combinar mejor las predicciones de múltiples modelos de aprendizaje automático de buen rendimiento.
- La biblioteca scikit-learn proporciona una implementación estándar de Stacking en Python.
- Cómo utilizar Stacking para modelos predictivos de regresión y clasificación.

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section1"></a>
# <font color="#004D7F"> 1. Algoritmo Stacked generalization</font>

- Stacked Generalization (Stacking) es un algoritmo el que se capacita a un clasificador para combinar a los clasificadores individuales.
- Su arquitectura implica un mínimo de dos niveles:
    - __Modelos de nivel 0 (Modelo base)__: los modelos se ajustan a los datos de entrenamiento y cuyas predicciones se comp
    - __Modelo de nivel 1 ( Metamodelo)__: modelo que aprende cómo combinar mejor las predicciones de los modelos.
- Los resultados de los modelos base utilizados como entrada al metamodelo pueden ser reales (regresión) o probabilidades/etiquetas clase (clasificación).
- El enfoque más común para preparar el conjunto de datos de entrenamiento para el metamodelo es mediante la validación cruzada de k veces de los modelos base, donde las predicciones fuera de pliegue se utilizan como base para el conjunto de datos de entrenamiento para el metamodelo.
- Stacking es apropiado cuando varios modelos tienen habilidades (pero estas son diferentes):
    - Es decir, las predicciones realizadas no están correlacionados o tienen una correlación baja
- Como tal, suele ser una buena idea utilizar una variedad de modelos que hacen suposiciones muy diferentes sobre cómo resolver la tarea de (seleccionamos modelos complejos y diversos)
- El metamodelo suele ser algoritmos de taxonomía linear, e.g., regresiones.

  
<figure><center>
  <img src="data/Stacked.png" width="450" height="250" alt="Gráfica">
  <figcaption><blockquote>Gradient Boosting Machine. Extraída de <a href="http://dx.doi.org/10.5220/0006431602160222">Demand Prediction Using Machine Learning Methods and Stacked Generalization</a></blockquote></figcaption>
</center></figure>


---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#004D7F"> 2. Especificaciones sobre Stacking</font>

La biblioteca de aprendizaje automático Python scikit-learn proporciona una implementación de Stacking para el aprendizaje automático a través de las clases `StackingRegressor` y `StackingClassifier`. 

- El uso del modelo requiere que especifique una lista de estimadores (modelos de nivel 0) y un estimador final (nivel 1 o metamodelo).
- Los de nivel 0 se indican mediante una lista con el parámetro `estimators`. También se le puede pasar un `Pipeline`.

```python
    ...
    models = [('lr',LogisticRegression()),('svm',make_pipeline(StandardScaler(),SVC()))]
    ensemble = StackingClassifier(estimators=models)
```

- El modelo de nivel 1 o metamodelo se proporciona a través del argumento `final_estimator` (Por defecto es `LinearRegression()` o `LogisticRegression()`.
- El conjunto de datos para el metamodelo se prepara mediante validación cruzada con el parámetro `cv` (por defecto es 5-fold), aunque también se puede llamar a un objeto `StratifiedKFold`. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`StackingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`StackingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.StackingRegressor.html)
</div>

<a id="section3"></a> 
# <font color="#004D7F"> 3. Stacking para Clasificación</font>

Veamos un ejemplo para un problema de clasificación

<a id="section31"></a> 
## <font color="#004D7F"> 3.1. Resultado de línea base</font>

Lo primero que vamos a ver el qué obtenemos de resultado en cada modelo de nivel 0.

Usamos la función `make_classification()` para crear un problema de clasificación binaria sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from numpy import mean
from numpy import std
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from matplotlib import pyplot

def get_dataset():
	X, y = make_classification(n_samples=5000, n_features=20, n_informative=10, n_redundant=10, random_state=1)
	return X, y

- Usamos los siguientes algoritmos para modelos de nivel 0: LoR, CART, NB, SVM y k-NN. 
- La función `get_models()` implementa esto y devuelve el diccionario de modelos a considerar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)
</div>

In [ ]:
def get_models():
	models = ???
	models['lr'] = ???
	models['knn'] = ???
	models['cart'] = ???
	models['svm'] = ???
	models['bayes'] = ???
	return models

Realizamos la evaluación con la función `evaluate_model()` en `RepeatedStratifiedKFold`.

In [ ]:
def evaluate_model(model, X, y):
	cv = ???
	scores = ???
	return scores

Vemos los resultados finales

In [ ]:
X, y = get_dataset()
models = get_models()
results, names = list(), list()
???:
	scores = ???
	???
	???
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:

pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<a id="section32"></a> 
## <font color="#004D7F"> 3.2. Integración Stacking</font>

- Combinamos estos cinco modelos en un solo modelo de conjunto mediante Stacking.
- Utilizamos LoR como metamodelo.
- La función `get_stacking()` define el modelo `StackingClassifier` combinando las predicciones de los modelos base utilizando una validación cruzada de 5 veces.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
</div>

In [ ]:
from sklearn.ensemble import StackingClassifier

def get_stacking():
	level0 = list()
	level0.append(('lr', LogisticRegression()))
	level0.append(('knn', KNeighborsClassifier()))
	level0.append(('cart', DecisionTreeClassifier()))
	level0.append(('svm', SVC()))
	level0.append(('bayes', GaussianNB()))
	level1 = ???
	model = ???
	return model

- Podemos incluir el conjunto de apilamiento en la lista de modelos para evaluar, junto con los modelos independientes con la función ya implementada `get_models()`.

In [ ]:
def get_models():
	models = dict()
	models['lr'] = LogisticRegression()
	models['knn'] = KNeighborsClassifier()
	models['cart'] = DecisionTreeClassifier()
	models['svm'] = SVC()
	models['bayes'] = GaussianNB()
	models['stacking'] = ???
	return models

Veamos los resultados.

In [ ]:
X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:

pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section4"></a> 
# <font color="#004D7F"> 4. Stacking para regresión</font>

Veamos ahora para un problema de regresión

<a id="section41"></a> 
## <font color="#004D7F"> 4.1. Resultado de línea base</font>

Usamos la función `make_regression()` para crear un problema de regresión sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from numpy import mean
from numpy import std
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from matplotlib import pyplot

def get_dataset():
	X, y = make_regression(n_samples=1000, n_features=20, n_informative=15, noise=0.1, random_state=1)
	return X, y

- Usamos los siguientes algoritmos para modelos de nivel 0: LoR, CART, NB, SVM y k-NN. 
- La función `get_models()` implementa esto y devuelve la lista de modelos a considerar.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVR`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html#sklearn.svm.SVR)
</div>

In [ ]:
def get_models():
	models = dict()
	models['knn'] = ???
	models['cart'] = ???
	models['svm'] = ???
	return models

Realizamos la evaluación con la función `evaluate_model()` en `RepeatedStratifiedKFold`.

In [ ]:
def evaluate_model(model, X, y):
	cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
	scores = cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
	return scores

Vemos los resultados finales

In [ ]:
X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:

pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

<a id="section42"></a> 
## <font color="#004D7F"> 4.2. Integración Stacking</font>

- Combinamos estos tres modelos en un solo modelo de conjunto mediante Stacking.
- Utilizamos LiR como metamodelo.
- La función `get_stacking()` define el modelo `StackingRegressor` combinando las predicciones de los modelos base utilizando una validación cruzada de 5 veces.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
</div>

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor

def get_stacking():
	level0 = list()
	level0.append(('knn', KNeighborsRegressor()))
	level0.append(('cart', DecisionTreeRegressor()))
	level0.append(('svm', SVR()))
	level1 = ???
	model = ???
	return model

- Podemos incluir el conjunto de apilamiento en la lista de modelos para evaluar, junto con los modelos independientes con la función ya implementada `get_models()`.

In [ ]:
def get_models():
	models = dict()
	models['knn'] = KNeighborsRegressor()
	models['cart'] = DecisionTreeRegressor()
	models['svm'] = SVR()
	models['stacking'] = ???
	return models

Veamos los resultados.

In [ ]:
X, y = get_dataset()
models = get_models()
results, names = list(), list()
for name, model in models.items():
	scores = evaluate_model(model, X, y)
	results.append(scores)
	names.append(name)
	print('>%s %.3f (%.3f)' % (name, mean(scores), std(scores)))

In [ ]:

pyplot.boxplot(results, labels=names, showmeans=True)
pyplot.show()

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="sectionEj"></a>
<h3><font color="#004D7F" size=6> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicios</font></h3>

Se proponen las siguientes actividades para consolidar el aprendizaje.

# <font color="#004D7F" size=5>Ejercicio 1</font>
__Hiperparámetros__. Explore diferentes configuraciones de hiperparámetros que veas en la librería sobre Stacking y comente los resultados. Utilice una estrategia de búsqueda.

# <font color="#004D7F" size=5>Ejercicio 2</font>
__Evaluar algoritmos__. Utilizar diferentes algoritmos a los que hemos usado en esta unidad y evaluar su impacto con Voting usando hiperparáemtros clave.

# <font color="#004D7F" size=5>Ejercicio 3</font>
__Datasets reales__. Busque un dataset original y verdadero (que no sea sintético) y evalúe el uso de los conceptos vistos en esta unidad. Los conjuntos de datos en pueden ser obtenidos del [repositorio de aprendizaje automático de UCI](https://archive.ics.uci.edu/).

# <font color="#004D7F" size=5>Ejercicio 4</font>
__Búsqueda de la mejor configuración__. Como se ha visto existen diferentes hiperparámetros que pueden ajustar nuestro modelo. Haga una búsqueda para un dataset real de cuales, entre un rango amplio de hiperparétros, maximizan la métrica. Puede utilizar una búsqueda aleatoria.

# <font color="#004D7F" size=5>Ejercicio 5</font>
__Modelos contribuyente__. Utilice otros modelos contribuyentes diferentes a los vistos y con diferentes hiperparámetros utilizando una estrategia de búsqueda.

# <font color="#004D7F" size=5>Ejercicio 6</font>
__Metaheurísticas__. En este clase hemos usado un algoritmo greedy para la poda y creciente. Use algoritmos más sofisticados para esto como, e.g., algoritmos basados en metaheurísticas. Seguramente tenga que probar con una batería mucho mayor de algoritmos.

# <font color="#004D7F" size=5>Ejercicio 7</font>
__Problema de regresión__. En este clase hemos trabajado un problema de clasificación. haga lo mismo con un problema de regresión y comente los resultados.

---

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>